In [2]:
# =============================================================================
# v4 MASTER PIPELINE
# Merges: ChatGPT Resilience + Claude Z-Score Gating + Gemini Lean Retrieval
# Includes: Ironclad Prompting & Regex Data Parsing
# =============================================================================
from __future__ import annotations

import json
import re
import sqlite3
import subprocess
import time
import sys
from pathlib import Path
from typing import Any, Literal, TypedDict

from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, END

try:
    from langgraph.checkpoint.memory import InMemorySaver as MemorySaver
except Exception:
    try:
        from langgraph.checkpoint.memory import MemorySaver
    except Exception:
        MemorySaver = None

# =============================================================================
# Block I — Locate project root and import project tools
# =============================================================================
PROJECT_ROOT = Path.cwd()
for _ in range(6):
    if (PROJECT_ROOT / "tools" / "llm_conversation.py").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent
else:
    raise RuntimeError("Could not locate project root containing tools/llm_conversation.py")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import tools.llm_conversation as _lc
from tools.llm_conversation import (
    _compact_report_for_llm,
    _parse_structured_queries,
    _sanitize_pmids,
)
from tools.pubmed_search import RetrievalConfig, evidence_to_text, search_pubmed

print("Imports OK")
print("Project root:", PROJECT_ROOT)

# =============================================================================
# Block II — Main configuration
# =============================================================================
DB_PATH = PROJECT_ROOT / "Actigraph_record.db"
AUDIENCE: Literal["expert", "doctor", "layperson"] = "doctor"
ANAMNESIS = ""  

ROW_KEY: tuple[str, str, str] | None = None

AGENT_CONFIG = {
    "data_summariser": {"model": "gemma4:12b", "temperature": 0.2},
    "pubmed_query_planner": {"model": "gemma4:12b", "fallback_model": "gemma4:12b", "temperature": 0.1},
    "relevance_judge": {"model": "gemma4:12b", "fallback_model": "qwen3.5:latest", "temperature": 0.0},
    "literature_synthesiser": {"model": "gemma4:12b", "fallback_model": "qwen3.5:latest", "temperature": 0.2},
    "symptom_metric_linker": {"model": "gemma4:12b", "temperature": 0.1},
    "report_writer": {"model": "gemma4:12b", "fallback_model": "qwen3.5:4b", "temperature": 0.1},
}

# FROM GEMINI: Lean retrieval configuration for speed
RETRIEVAL_CFG = RetrievalConfig(
    retmax_per_query=25,
    keep_per_query=8,
    max_total_items=25,
    years_back=15,
    humans_only=True,
    adults_only=(AUDIENCE == "doctor"),
)

# THE FIX: Lowered Z-Score threshold to catch 20%+ changes (e.g. 16m sleep drop)
CONSTRUCT_Z_THRESHOLD = 0.2
MAX_CONSTRUCTS = 6

# FROM CHATGPT: The Soft Floor to prevent data starvation
MIN_ITEMS_AFTER_JUDGE = 8

USE_TUNING_MEMORY = True
MEMORY_PATH = PROJECT_ROOT / "Development" / "llm_pipeline_tuning_memory.json"
THREAD_ID = "llm-pipeline-v4"

# =============================================================================
# Block III — Load latest analysis row from SQLite
# =============================================================================
con = sqlite3.connect(DB_PATH)
rows = con.execute(
    "SELECT username, period_id_1, period_id_2, audience, model, created_at "
    "FROM ai_analysis_runs ORDER BY created_at DESC"
).fetchall()

if not rows:
    raise RuntimeError("No rows found in ai_analysis_runs")

if ROW_KEY is None:
    ROW_KEY = (rows[0][0], rows[0][1], rows[0][2])
print(f"\nSelected: username={ROW_KEY[0]} P1={ROW_KEY[1]} P2={ROW_KEY[2]}")

raw = con.execute(
    "SELECT json_input FROM ai_analysis_runs "
    "WHERE username=? AND period_id_1=? AND period_id_2=? "
    "ORDER BY created_at DESC LIMIT 1",
    ROW_KEY,
).fetchone()
con.close()

report_data = json.loads(raw[0])
compact_report = _compact_report_for_llm(report_data)

# =============================================================================
# Block IV — Regex Construct Mapping (The Parser Fix)
# =============================================================================
METRIC_CONSTRUCT_MAP: dict[str, tuple[str, str]] = {
    "SRI":         ("sleep irregularity",                  "sleep regularity"),
    "IS":          ("circadian rhythm fragmentation",      "circadian stability"),
    "IV":          ("circadian rhythm stability",          "intradaily variability fragmentation"),
    "RA":          ("dampened circadian amplitude",        "robust circadian rhythm"),
    "L5":          ("restless nighttime activity",         "consolidated rest period"),
    "M10":         ("low daytime activity",                "high daytime activity"),
    "Sleep Duration": ("short sleep duration",             "long sleep duration"),
    "CPD mid sleep":  ("advanced sleep phase",             "delayed sleep phase"),
    "SE":          ("poor sleep efficiency",               "high sleep efficiency"),
    "WASO":        ("fragmented sleep",                    "consolidated sleep"),
}

def derive_constructs(compact_report_text: str, z_threshold: float = CONSTRUCT_Z_THRESHOLD) -> list[dict]:
    """
    Bypasses fragile JSON parsing and mathematically extracts anomalies 
    directly from the compact_report string using Regex.
    """
    out: list[dict] = []
    
    # Looks for: "Metric Name: P1=100.0, P2=50.0"
    pattern = r"([^|]+?)\s*:\s*P1=([-\d\.]+),\s*P2=([-\d\.]+)"
    matches = re.findall(pattern, compact_report_text)
    
    for metric_name, p1_str, p2_str in matches:
        metric_name = metric_name.strip()
        p1, p2 = float(p1_str), float(p2_str)
        
        if p1 == 0: 
            continue
            
        # Calculate relative percentage change
        delta_z = (p2 - p1) / abs(p1)
        
        if abs(delta_z) >= z_threshold:
            # Map metric name to ontology
            key = metric_name
            for canon in METRIC_CONSTRUCT_MAP:
                if canon.lower() in metric_name.lower():
                    key = canon
                    break
            else: 
                continue # Skip if no clinical mapping exists
            
            down, up = METRIC_CONSTRUCT_MAP[key]
            out.append({
                "metric": metric_name,
                "direction": "decreased" if delta_z < 0 else "increased",
                "magnitude_z": round(delta_z, 2),
                "construct": down if delta_z < 0 else up,
            })
            
    out.sort(key=lambda x: abs(x["magnitude_z"]), reverse=True)
    return out

# =============================================================================
# Block V — Modified Prompts (The "v4" Fixes)
# =============================================================================
AGENT1_SYSTEM = _lc._AGENT1_SYSTEM
RELEVANCE_SYSTEM = _lc._RELEVANCE_SYSTEM
AGENT6_SYSTEM = _lc._AGENT6_SYSTEM

# 1. Query Planner: Gemini's defensive "NO QUOTES" rule + Claude's Constructs
AGENT2_SYSTEM = """You are a research query planner for clinical actigraphy reports.
You receive a list of CLINICAL CONSTRUCTS derived from this patient's significant metric anomalies.

Your job: produce ONE PubMed query intent per construct.

CRITICAL PUBMED SEARCH RULES:
1. NO QUOTATION MARKS: NEVER put quotes around your search terms. It breaks the PubMed API.
2. KEEP IT SIMPLE: Use 2 to 3 broad keywords per query. 
3. TARGET OUTCOMES: Target the clinical construct and its meaningful health outcomes.

Output: JSON lines, one object per query:
{"topic": "...", "population": "...", "context": "...", "expected_link": "..."}
"""

# 2. Synthesiser: Claude's rigid formatting (Phi4 loves this)
AGENT4_SYSTEM = """You are a clinical literature synthesiser writing for a busy doctor.
You receive a list of CLINICAL CONSTRUCTS with matched PubMed evidence.

For EACH construct, write ONE short paragraph (max 60 words) that:
1. Explains what the metric change indicates clinically.
2. Cites 1-3 PMIDs linking that construct to outcomes/mechanisms.

Hard rules:
- ONE paragraph per construct.
- Format exactly as: **Construct name** — paragraph text (PMID xxxxxxxx).
- If no evidence is matched, write: **Construct name** — (no direct evidence retrieved).
"""

# 3. Report Writer: The Ironclad Patient-Anchoring Constraint
AGENT5_AUDIENCES = dict(_lc._AGENT5_AUDIENCES)
AGENT5_AUDIENCES["doctor"] = """You are a clinical data translator. Your ONLY job is to write a patient-specific actigraphy report for a doctor.

CRITICAL RULES:
1. NO ESSAYS. NEVER write a general introduction, conclusion, or literature review.
2. NO GENERAL STUDIES. Do not summarize general population studies (e.g., "A study of 72,000 people...").
3. PATIENT FIRST. EVERY single bullet point MUST start with the patient's actual numbers from the 'Actigraphy Context'.

You MUST output EXACTLY this format, and nothing else:

## Bottom Line
[One sentence combining the patient's most severe metric anomaly (e.g., sleep dropping to 16 mins) with its primary clinical risk based on the literature.]

## Key Findings & Literature Context
- **[Metric Name] ([Patient's Exact P1 to P2 Numbers]):** [What this means clinically for THIS patient]. Evidence suggests this specific pattern relates to [1 concise sentence connecting the finding to a retrieved paper] (PMID xxxxxxxx).
- **[Metric Name] ([Patient's Exact P1 to P2 Numbers]):** [Repeat for up to 3 total major anomalies].

## Suggested Next Steps
- [Actionable step 1 for the clinician based strictly on these anomalies]
- [Actionable step 2]
"""

# =============================================================================
# Block VI — State & Utility Functions
# =============================================================================
class PipelineState(TypedDict, total=False):
    compact_report: str
    audience: str
    anamnesis: str
    tuning_memory: str
    data_summary: str
    constructs: list[dict]
    search_queries: list[dict]
    evidence_items: list[dict]
    raw_abstracts: str
    pmid_list: list[str]
    lit_summary: str
    symptom_metric_table: str
    final_report: str
    claim_to_pmid_map: dict[str, list[str]]
    trace: list[dict[str, Any]]

def chat_node(node: str, system: str, user: str, *, json_mode: bool = False) -> str:
    cfg = AGENT_CONFIG[node]
    model_names = [cfg["model"]]
    if cfg.get("fallback_model"): model_names.append(cfg["fallback_model"])

    last_error = None
    for model_name in model_names:
        try:
            kwargs = {"model": model_name, "temperature": cfg.get("temperature", 0.2)}
            if json_mode: kwargs["format"] = "json"
            llm = ChatOllama(**kwargs)
            response = llm.invoke([SystemMessage(content=system), HumanMessage(content=user)])
            return str(response.content).strip()
        except Exception as exc:
            last_error = exc
    raise RuntimeError(f"{node} failed") from last_error

# =============================================================================
# Block VII — LangGraph Nodes
# =============================================================================
def data_summariser(state: PipelineState) -> PipelineState:
    user = f"# Actigraphy Metrics\n{state['compact_report']}\n\nSummarise these metrics for a downstream researcher."
    summary = chat_node("data_summariser", AGENT1_SYSTEM, user)
    return {**state, "data_summary": summary}

def construct_mapper(state: PipelineState) -> PipelineState:
    # PASSING THE TEXT INSTEAD OF THE JSON DICT
    constructs = derive_constructs(state['compact_report'], z_threshold=CONSTRUCT_Z_THRESHOLD)
    return {**state, "constructs": constructs}

def pubmed_query_planner(state: PipelineState) -> PipelineState:
    constructs = state.get("constructs", [])[:MAX_CONSTRUCTS]
    if not constructs:
        queries = [{"topic": "circadian rhythm disruption", "population": "humans", "context": "health outcomes"}]
        return {**state, "search_queries": queries}

    construct_block = "\n".join(f"- {c['metric']} {c['direction']} → {c['construct']}" for c in constructs)
    user = f"# Detected metric changes\n{construct_block}\n\nGenerate ONE PubMed query intent per construct."
    raw = chat_node("pubmed_query_planner", AGENT2_SYSTEM, user)
    queries = _parse_structured_queries(raw)
    
    return {**state, "search_queries": queries if queries else [{"topic": c["construct"], "population": "humans", "context": "outcomes"} for c in constructs]}

def pubmed_retriever(state: PipelineState) -> PipelineState:
    items = search_pubmed(state.get("search_queries", []), config=RETRIEVAL_CFG)
    
    # ChatGPT Fallback Loop
    if len(items) < MIN_ITEMS_AFTER_JUDGE:
        fallback_queries = [{"topic": c["construct"], "population": "humans", "context": "clinical outcomes"} for c in state.get("constructs", [])]
        items += search_pubmed(fallback_queries, config=RETRIEVAL_CFG)
        
    return {**state, "evidence_items": items, "raw_abstracts": evidence_to_text(items)}

def relevance_judge(state: PipelineState) -> PipelineState:
    items = state.get("evidence_items", [])
    if not items: return state

    user = f"Clinical summary:\n{state.get('data_summary', '')}\n\nEvidence items:\n{json.dumps(items, ensure_ascii=False)}"
    raw = chat_node("relevance_judge", RELEVANCE_SYSTEM, user, json_mode=True)
    
    # Extract PMIDs
    try: parsed = json.loads(raw)
    except: parsed = {}
    keep_pmids = {str(x) for x in (parsed.get("keep_pmids") or parsed.get("pmids") or [])}
    
    # ChatGPT Soft Floor implementation
    if keep_pmids and len(keep_pmids) >= MIN_ITEMS_AFTER_JUDGE:
        items = [x for x in items if str(x.get("pmid")) in keep_pmids]
    else:
        items = items[: max(MIN_ITEMS_AFTER_JUDGE, min(len(items), 18))]

    return {**state, "evidence_items": items, "pmid_list": [str(x.get("pmid")) for x in items if x.get("pmid")]}

def literature_synthesiser(state: PipelineState) -> PipelineState:
    user = (
        f"# Constructs\n{json.dumps(state.get('constructs', [])[:MAX_CONSTRUCTS])}\n\n"
        f"# Abstracts\n{state.get('raw_abstracts', '')}\n\n"
        "Write ONE short paragraph per construct following the system prompt exactly."
    )
    summary = chat_node("literature_synthesiser", AGENT4_SYSTEM, user)
    return {**state, "lit_summary": summary}

def symptom_metric_linker(state: PipelineState) -> PipelineState:
    if not state.get("anamnesis"): return state
    user = f"# Patient Anamnesis\n{state['anamnesis']}\n\n# Data Summary\n{state['data_summary']}\n\nProduce the correlation table."
    return {**state, "symptom_metric_table": chat_node("symptom_metric_linker", AGENT6_SYSTEM, user)}

def report_writer(state: PipelineState) -> PipelineState:
    audience = state.get("audience", "doctor")
    system_prompt = AGENT5_AUDIENCES.get(audience, AGENT5_AUDIENCES["doctor"])
    
    user = (
        f"# Actigraphy Context (The Patient's Data)\n{state['compact_report']}\n\n"
        f"# Literature Synthesis (From Agent 4)\n{state.get('lit_summary', '')}\n\n"
        "Write the final report now. "
        "CRITICAL: You MUST use the EXACT markdown template provided in your system instructions. "
        "Do NOT write an essay. Start immediately with '## Bottom Line'."
    )
    
    report = chat_node("report_writer", system_prompt, user)
    
    # We still sanitize the PMIDs here using the state list to ensure safety, 
    # even though we didn't feed the raw JSON to the LLM.
    allowed = {str(p) for p in state.get("pmid_list", [])}
    report = _sanitize_pmids(report, allowed)
    
    if allowed:
        report += "\n\n---\nReferences:\n" + "\n".join(f"{i+1}. PMID {p}" for i, p in enumerate(list(allowed)[:10]))
        
    return {**state, "final_report": report}

# =============================================================================
# Block VIII — Build & Run Graph
# =============================================================================
graph = StateGraph(PipelineState)
graph.add_node("data_summariser", data_summariser)
graph.add_node("construct_mapper", construct_mapper)
graph.add_node("pubmed_query_planner", pubmed_query_planner)
graph.add_node("pubmed_retriever", pubmed_retriever)
graph.add_node("relevance_judge", relevance_judge)
graph.add_node("literature_synthesiser", literature_synthesiser)
graph.add_node("symptom_metric_linker", symptom_metric_linker)
graph.add_node("report_writer", report_writer)

graph.set_entry_point("data_summariser")
graph.add_edge("data_summariser", "construct_mapper")
graph.add_edge("construct_mapper", "pubmed_query_planner")
graph.add_edge("pubmed_query_planner", "pubmed_retriever")
graph.add_edge("pubmed_retriever", "relevance_judge")
graph.add_edge("relevance_judge", "literature_synthesiser")

def route_after_synthesis(state: PipelineState) -> str:
    return "symptom_metric_linker" if state.get("anamnesis", "").strip() else "report_writer"

graph.add_conditional_edges("literature_synthesiser", route_after_synthesis, {"symptom_metric_linker": "symptom_metric_linker", "report_writer": "report_writer"})
graph.add_edge("symptom_metric_linker", "report_writer")
graph.add_edge("report_writer", END)

AGENT_GRAPH = graph.compile()

print("\nRunning v4 Master Pipeline...")
t0 = time.time()
final_state = AGENT_GRAPH.invoke({
    "compact_report": compact_report,
    "audience": AUDIENCE,
    "anamnesis": ANAMNESIS.strip(),
})

print(f"Graph finished in {time.time() - t0:.1f}s")
print("\n# Final v4 Report\n")
print(final_state.get("final_report", ""))

Imports OK
Project root: /Users/arahjou/Documents/APP_CIRCADIAN_MEDICINE_v7

Selected: username=admin P1=ID-001 P2=ID-002

Running v4 Master Pipeline...


KeyboardInterrupt: 